In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from peakstay.api import fetch_all_shelters, load_raw, save_raw
from peakstay.cleaning import build_dataframe, clean_altitude, keep_staffed_open, normalize_text


API CAI: 764 rifugi, 77 pagine da 10

In [3]:
RAW_PATH = Path("../data/raw/shelters.json")

if RAW_PATH.exists():
    shelters = load_raw(RAW_PATH)
else:
    shelters = fetch_all_shelters()
    save_raw(shelters, RAW_PATH)

print(len(shelters))

764


In [4]:
df = build_dataframe(shelters)
df = clean_altitude(df)
df = keep_staffed_open(df)
df = normalize_text(df)
print(df.shape)

(282, 139)


In [7]:
print(df["category"].unique())

<ArrowStringArray>
['E', 'D', 'B', 'C', 'A']
Length: 5, dtype: str


In [5]:
cols = ["region_geo", "ownerRegion_geo", "municipality_geo",
        "massif_geo", "locality_geo", "altitude_geo"]
print(df[cols].notna().sum())

print("comuni distinti:", df["municipality_geo"].nunique())
print("massicci distinti:", df["massif_geo"].nunique())

region_geo          282
ownerRegion_geo     282
municipality_geo    282
massif_geo          281
locality_geo        282
altitude_geo        282
dtype: int64
comuni distinti: 194
massicci distinti: 32


In [11]:
per_mun = df["municipality_geo"].value_counts()
print((per_mun == 1).sum(), "comuni con un solo rifugio su", len(per_mun))
print(per_mun.head(10))

print(df["massif_geo"].value_counts().describe())

both = df.dropna(subset=["region_geo", "ownerRegion_geo"])
diff = both[both["region_geo"] != both["ownerRegion_geo"]]
print(len(diff))
print(diff[["title", "region_geo", "ownerRegion_geo"]].head(10))

service_cols = [c for c in df.columns if c.endswith("_service")]
print(len(service_cols))
print(df[service_cols].notna().sum().sort_values(ascending=False).head(15))

260 comuni con un solo rifugio su 406
municipality_geo
Courmayeur              18
Macugnaga               13
Alagna Valsesia         12
Auronzo di Cadore       12
Valfurva                 9
Val Masino               9
Valdieri                 9
Tarvisio                 9
Cogne                    8
Chiesa in Valmalenco     7
Name: count, dtype: int64
count     62.000000
mean      12.193548
std       20.401574
min        1.000000
25%        1.000000
50%        4.000000
75%       15.750000
max      118.000000
Name: count, dtype: float64
65
                                        title           region_geo  \
2                         CITTA' DI GALLARATE             Piemonte   
7                                MUSSO BIAGIO             Svizzera   
11  SELLA QUINTINO al Felik (NUOVO E VECCHIO)        Valle d'Aosta   
13                            CITTA' DI LUINO             Piemonte   
16                           CITTA' DI CANTU'  Trentino-Alto Adige   
19                              BALZOL

In [7]:
service_cols = [c for c in df.columns if c.endswith("_service")]
print(len(service_cols))
print(pd.unique(df[service_cols].values.ravel()))

43
['1' '12' '82' '0' '6' '3' nan 'Segnale GSM' '9' '8' 'Assente' 'Nessuna'
 '10' '2' '176' '186' '5' 'Tim' 'Buono' '20' '16' '104' '48' '54' '142'
 '146' '106' '114' '90' 'Nessuno' '7' '66' '57' '4' '40' '70' '11'
 'Tim, Vodafone' '150' '170' '14' '122' '28' '15' '60' '35' '192' '18'
 '200' 'Tim, Vodafone, Wind' '24' '38' '25' '36' '30' '39' '45' '34' '37'
 '32' 'mostre, incontri geologici e astronomici' '50' '56' '69' '76' '100'
 '74' '64' '49' 'Soci CAI' '159' '180' '140' 'tim, vodafone' 'Vodafone'
 'agenzie viaggi' 'Instabile' '158' '173' '13' '120' '80' '96' '92' '23'
 '99' '67' '42' '33' '26' '21' '94' '93' '19' '46' '138' '71'
 'avvicinamento arrampicata, corsi di fotografia digitale, proiezioni di video e foto, attività per bambini'
 '111' '68' '84' '47' '61' '.0' '22' '29' '51' '52' '65' 'Vodafone, Wind'
 '108' '98' '58' 'Sufficiente' '89' '102' '155' '151' '109' '125' '44'
 '41' '136' '132' '63' '116' '85' '62' '31' '78' '75' 'Wind'
 'concerti in quota, presentazione libri' '

In [23]:
from peakstay.recommender import geo_score

print(df["region_geo"].unique())
s = geo_score(df, "Valle d'Aosta", "Courmayeur")
print(s.value_counts())

<ArrowStringArray>
[             'Piemonte',         'Valle d'Aosta',             'Lombardia',
   'Trentino-Alto Adige',                'Veneto',               'Abruzzo',
 'Friuli Venezia Giulia',                 'Lazio',        'Emilia-Romagna',
               'Toscana',               'Sicilia',               'Liguria']
Length: 12, dtype: str
0.000000    257
0.666667     12
0.333333      8
1.000000      5
Name: count, dtype: int64


In [26]:
from peakstay.recommender import recommend

res = recommend(df, "Valle d'Aosta", "Courmayeur", target_altitude=2500)
res[["title", "region_geo","municipality_geo", "massif_geo", "altitude_geo", "score"]]


,title,region_geo,municipality_geo,massif_geo,altitude_geo,score
29,DALMAZZI CESARE AL TRIOLET,Valle d'Aosta,Courmayeur,Alpi Graie,2590,0.986133
103,SOLDINI MONTANARO ELISABETTA,Valle d'Aosta,Courmayeur,Alpi Graie,2195,0.953005
7,GONELLA FRANCESCO (NUOVO E VECCHIO),Valle d'Aosta,Courmayeur,Alpi Graie,3072,0.911864
202,MONTE BIANCO,Valle d'Aosta,Courmayeur,Alpi Graie,1700,0.876733
5,TORINO (NUOVO E VECCHIO),Valle d'Aosta,Courmayeur,Alpi Graie,3375,0.865177
46,DEFFEYES ALBERTO,Valle d'Aosta,La Thuile,Alpi Graie,2498,0.766359
30,SELLA VITTORIO,Valle d'Aosta,Cogne,Alpi Graie,2588,0.753107
19,VITTORIO EMANUELE (NUOVO E VECCHIO),Valle d'Aosta,Valsavarenche,Alpi Graie,2732,0.730919
69,CRETE SÈCHE,Valle d'Aosta,Bionaz,Alpi Pennine,2400,0.517925
16,CAPANNA AOSTA,Valle d'Aosta,Bionaz,Alpi Pennine,2788,0.488957


In [25]:
for col in ["region_geo", "municipality_geo", "massif_geo", "locality_geo"]:
    raw = df[col].nunique()
    clean = df[col].str.strip().str.lower().nunique()
    print(col, raw, clean)

region_geo 12 12
municipality_geo 194 194
massif_geo 32 32
locality_geo 279 279


In [10]:
for col in ["region_geo", "municipality_geo", "massif_geo", "locality_geo"]:
    s = df[col].dropna()
    grouped = s.groupby(s.str.strip().str.lower()).unique()
    variants = grouped[grouped.str.len() > 1]
    if not variants.empty:
        print(f"--- Varianti trovate in {col} ---")
        for v in variants:
            print(list(v))


In [22]:
for col in ["type", "status", "category"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

keywords = ["open", "apert", "custod", "guard", "stagion", "period", "attiv", "chius"]
candidates = [c for c in df.columns if any(k in c.lower() for k in keywords)]
print(candidates)

print(pd.Series([s["published"] for s in shelters]).value_counts())
print(pd.Series([s["deleted_at"] is not None for s in shelters]).value_counts())

type
type
Rifugio custodito    282
Name: count, dtype: int64

status
status
In attività    282
Name: count, dtype: int64

category
category
C    127
D    103
A     34
E     10
B      8
Name: count, dtype: int64

['openingTime_contact', 'attivita_culturali_service', 'posti_attivita_culturali_service']
True    764
Name: count, dtype: int64
False    764
Name: count, dtype: int64


In [24]:
sorted(df["region_geo"].dropna().unique())


['Abruzzo',
 'Emilia-Romagna',
 'Friuli Venezia Giulia',
 'Lazio',
 'Liguria',
 'Lombardia',
 'Piemonte',
 'Sicilia',
 'Toscana',
 'Trentino-Alto Adige',
 "Valle d'Aosta",
 'Veneto']

In [27]:
print("Min:", df["altitude_geo"].min())
print("Max:", df["altitude_geo"].max())
print("Mediana:", df["altitude_geo"].median())


Min: 611
Max: 4554
Mediana: 1995.5


In [ ]:
print("Max:", df["altitude_geo"].max())

In [1]:
per_mun = df["region_geo"].value_counts()
print(per_mun.head(10))

print(df["region_geo"].value_counts().describe())


NameError: name 'df' is not defined